In [ ]:
from src.analysis.trackers import (
    tracker_prevalence_by_profile,
    tracker_frequency_table,
    jaccard_similarity_matrix,
    differential_trackers,
)
from src.analysis.statistics import pairwise_battery, chi_square_batch
from src.viz.tracker_plots import (
    apply_style, plot_prevalence_bars, plot_tracker_heatmap,
    plot_jaccard_matrix, plot_differential_trackers,
)

apply_style()

# Headline figure
prev = tracker_prevalence_by_profile()
plot_prevalence_bars(prev, save_path='fig01_tracker_breadth.pdf')

# Tracker heatmap
freq = tracker_frequency_table(top_n=25)
plot_tracker_heatmap(freq, save_path='fig02_tracker_heatmap.pdf')

# Statistical evidence
stats_df = pairwise_battery('unique_hosts_per_visit')
print(stats_df.to_string(index=False))

# Per-profile differential analysis
for profile in ['shopping', 'news', 'health']:
    diff = differential_trackers(profile, 'control', min_visits=5)
    plot_differential_trackers(diff, f'{profile} vs control',
                               save_path=f'fig03_diff_{profile}.pdf')
    
    # Confirm top differentials are statistically significant
    chi_results = chi_square_batch(profile, 'control',
                                    diff.head(30)['etld1'].tolist())
    print(f"\nSignificant trackers for {profile}:")
    print(chi_results[chi_results['reject_null']]
          [['comparison', 'p_value_corrected', 'effect_size']]
          .to_string(index=False))